In [39]:
from sympy import Symbol
from haarpy import haar_integral_unitary
import re
from IHU_source import *
def str_to_expr_dict(string):
    # parse the string into a dict of the form { (name, rep): {"in": lst[(leg, in indices)], "out": lst[(leg, out indices)]} }
    # e.g. X(ij,kl) U*(jk,li) -> { ("X", 1): {"in": [(1, "i"), (2, "j")], "out": [(1, "k"), (2, "l")]}, ("U*", 1): {"in": [(1, "j"), (2, "k")], "out": [(1, "l"), (2, "i")]}}
    symbs = 'abcdefghijklmnopqrstuvwxyz'
    idx = 0
    expr_dict = dict()
    string = string.replace('\n', '')
    terms = re.findall(r'[\w*]+\([^\)]+\)', string)
    for term in terms:
        name = term.split('(')[0]
        in_out = term.split('(')[1].split(')')[0]
        
        split = in_out.split(',')
        in_lst = split[0]
        if ' ' in in_lst:
            in_lst = in_lst.strip().split(' ')
        out_lst = split[1] if len(split) > 1 else ""
        if ' ' in out_lst:
            out_lst = out_lst.strip().split(' ')
        rep = 1
        print(name, in_lst, out_lst)
        while (name, rep) in expr_dict:
            rep += 1
        expr_dict[(name, rep)] = {"in": [], "out": []}
        for leg, symb in enumerate(in_lst):
            if symb != '.':
                expr_dict[(name, rep)]["in"].append( (leg+1, symb) )
        for leg, symb in enumerate(out_lst):
            if symb != '.':
                expr_dict[(name, rep)]["out"].append( (leg+1, symb) )
    return expr_dict
def to_expr_dict(edges):
    symbs = 'abcdefghijklmnopqrstuvwxyz'
    idx = 0
    expr_dict = dict() # key: (name, rep), value: dict("in":lst[(leg, in indices)], "out":lst[(leg, out indices)])
    for idxa,idxb in edges:
        assert len(idxa) == 4 and len(idxb) == 4
        # name: name of the tensor, e.g. X, U, U*
        # rep: repeat number
        # leg: leg number starting from 1
        # direction: "in" or "out"
        name, rep, direction, leg = idxa
        name2, rep2, direction2, leg2 = idxb
        if (name, rep) not in expr_dict:
            expr_dict[(name, rep)] = {"in": [], "out": []}
        if (name2, rep2) not in expr_dict:
            expr_dict[(name2, rep2)] = {"in": [], "out": []}
        expr_dict[(name, rep)][direction].append( (leg, symbs[idx]) )
        expr_dict[(name2, rep2)][direction2].append( (leg2, symbs[idx]) )
        idx += 1
    return expr_dict
def to_lst(idx_lst:list[tuple]):
    idx_lst.sort(key=lambda x: x[0], )
    res = []
    curr_idx = 1
    if not idx_lst:
        return []
    for leg, symb in idx_lst:
        while curr_idx < leg:
            res.append('.') # open leg
            curr_idx += 1
        res.append(symb)
        curr_idx += 1
    return res
def to_str(expr_dict):
    res = ""
    for (name, rep), directions in expr_dict.items():
        in_lst = to_lst(directions['in'])
        out_lst = to_lst(directions['out'])
        res += f"{name}({''.join(out_lst)},{''.join(in_lst)}) "
    return res
def to_edge(expr_dict):
    edges = dict()
    res=[]
    for (name, rep), directions in expr_dict.items():
        for direction, lst in directions.items():
            for leg, symb in lst:
                if symb not in edges:
                    edges[symb] = [name, rep, direction, leg]
                else:
                    src = edges[symb]
                    dst = [name, rep, direction, leg]
                    del edges[symb]
                    res.append( [src, dst] )
    return res
d = symbols('d')
dic = str_to_expr_dict("A(a) B(b) U(a,i) U(b,j) X(ij,kl) U*(k,c) U*(l,d) C(c) D(d)")
print(dic)
input_ = [to_edge(dic),1]
print(input_)
edge = integrateHaarUnitary(input_,["U",[d],[d],d])
for term in edge:
    expr, coeff = term
    print(to_str(to_expr_dict(expr)), coeff)
# e1 = [["X", 1, "out", 1], ["U", 1, "in", 1]]
# e2 = [["U*", 1, "out", 1], ["X", 1, "in", 1]]
# e3 = [["X", 1, "out", 2], ["U", 2, "in", 1]]
# e4 = [["U*", 2, "out", 1], ["X", 1, "in", 2]]
# g = [e1, e2, e3, e4]
# [[['U', 1, 'out', 1], ['X', 1, 'in', 1]], 
#  [['U', 2, 'out', 1], ['X', 1, 'in', 2]], 
#  [['X', 1, 'out', 1], ['U*', 1, 'in', 1]], 
#  [['X', 1, 'out', 2], ['U*', 2, 'in', 1]]
#  ]
# lst = to_expr_dict(g)
# print(to_str(lst))
# # gw = [g,1]
# # print(gw)
# # visualizeTN(gw)


# Eg = integrateHaarUnitary(gw, ["U", [d], [d], d])
# print(Eg)

# visualizeTN(Eg)


A a 
B b 
U a i
U b j
X ij kl
U* k c
U* l d
C c 
D d 
{('A', 1): {'in': [(1, 'a')], 'out': []}, ('B', 1): {'in': [(1, 'b')], 'out': []}, ('U', 1): {'in': [(1, 'a')], 'out': [(1, 'i')]}, ('U', 2): {'in': [(1, 'b')], 'out': [(1, 'j')]}, ('X', 1): {'in': [(1, 'i'), (2, 'j')], 'out': [(1, 'k'), (2, 'l')]}, ('U*', 1): {'in': [(1, 'k')], 'out': [(1, 'c')]}, ('U*', 2): {'in': [(1, 'l')], 'out': [(1, 'd')]}, ('C', 1): {'in': [(1, 'c')], 'out': []}, ('D', 1): {'in': [(1, 'd')], 'out': []}}
[[[['A', 1, 'in', 1], ['U', 1, 'in', 1]], [['B', 1, 'in', 1], ['U', 2, 'in', 1]], [['U', 1, 'out', 1], ['X', 1, 'in', 1]], [['U', 2, 'out', 1], ['X', 1, 'in', 2]], [['X', 1, 'out', 1], ['U*', 1, 'in', 1]], [['X', 1, 'out', 2], ['U*', 2, 'in', 1]], [['U*', 1, 'out', 1], ['C', 1, 'in', 1]], [['U*', 2, 'out', 1], ['D', 1, 'in', 1]]], 1]
A(,a) C(,a) B(,b) D(,b) X(cd,cd)  1/(d**2 - 1)
A(,a) D(,a) B(,b) C(,b) X(cd,cd)  -1/(d**3 - d)
A(,a) C(,a) B(,b) D(,b) X(cd,dc)  -1/(d**3 - d)
A(,a) D(,a) B(,b) C(,b) X(cd,dc)  1

In [ ]:
e1 = [["A", 1, "out", 1], ["U", 1,  "in", 1]]
e2 = [["A", 1, "out", 2], ["U", 1, "in", 2]]
e3 = [["U*", 1, "out", 1], ["A", 1, "in", 1]]
e4 = [["U*", 1, "out", 2], ["A", 1, "in", 2]]
e5 = [["U", 1, "out", 2], ["U*", 1, "in", 2]]
g = [e1, e2, e3, e4, e5]
gw = [g,1]
print(to_str(to_expr_dict(g)))

k,n = symbols('k,n')
Eg = integrateHaarUnitary(gw, ["U", [n, k], [n, k], n*k])
for term in Eg:
    expr, coeff = term
    print(to_str(to_expr_dict(expr)), coeff)

A(ab,cd) U(.e,ab) U*(cd,.e) 
@U*(.,a) @U(a,.) A(bc,bc)  1/n


In [40]:
d,k,n = symbols('d,k,n')
e1 = [["U*", 2, "out", 1], ["U", 1, "in", 1]]
e2 = [["U*", 1, "out", 1], ["U", 2, "in", 1]]
e3 = [["U", 1, "out", 1], ["U*", 1, "in", 1]]
e4 = [["U", 2, "out", 1], ["U*", 2, "in", 1]]
e5 = [["U", 1, "out", 2], ["U*", 2, "in", 2]]
e6 = [["U", 2, "out", 2], ["U*", 1, "in", 2]]
g = [e1, e2, e3, e4, e5, e6]
gw = [[g, 1/(d* k)]]
print(to_str(to_expr_dict(g)))
Eg = integrateHaarUnitary(gw, ["U", [d], [n, k], n*k])
# print(Eg)
# #visualizeTN(Eg)
for term in Eg:
    expr, coeff = term
    print(to_str(to_expr_dict(expr)), coeff)
from sympy import limit, oo
t = symbols('t')
Egt =  Eg[0][1].subs(d,t*k*n)
Egt_lim = simplify(limit(Egt,n,oo))
print(Egt_lim)

U*(a,de) U(ce,a) U*(b,cf) U(df,b) 
 (-d*n + k*(d*k*n + n**2 - 1))/(k*(k**2*n**2 - 1))
(k**2*t - t + 1)/k**2


In [45]:
expr = """  U(a1 a2 a3 a4, a1p a2p a3p a4p) R(a1p a2p a3p a4p, a1q a2q a3q a4q) U*(a1q a2q a3q a4q, b1 b2 a3 a4) 
            U(b1 b2 b3 b4, b1p b2p b3p b4p) R(b1p b2p b3p b4p, b1q b2q b3q b4q) U*(b1q b2q b3q b4q, a1 a2 b3 b4)
            U(c1 c2 c3 c4, c1p c2p c3p c4p) R(c1p c2p c3p c4p, c1q c2q c3q c4q) U*(c1q c2q c3q c4q, d1 c2 d3 c4)
            U(d1 d2 d3 d4, d1p d2p d3p d4p) R(d1p d2p d3p d4p, d1q d2q d3q d4q) U*(d1q d2q d3q d4q, c1 d2 c3 d4)"""
edges = to_edge(str_to_expr_dict(expr))
print(edges)
d1,d2,d3,d4 = symbols('d1 d2 d3 d4')
res = integrateHaarUnitary([edges, 1], ["U", [d1,d2,d3,d4], [d1,d2,d3,d4], d1*d2*d3*d4])
sum_ = 0
for term in res:
    expr, coeff = term
    print(to_str(to_expr_dict(expr)), coeff)
    sum_ += coeff
print(sum_)

U ['a1', 'a2', 'a3', 'a4'] ['a1p', 'a2p', 'a3p', 'a4p']
R ['a1p', 'a2p', 'a3p', 'a4p'] ['a1q', 'a2q', 'a3q', 'a4q']
U* ['a1q', 'a2q', 'a3q', 'a4q'] ['b1', 'b2', 'a3', 'a4']
U ['b1', 'b2', 'b3', 'b4'] ['b1p', 'b2p', 'b3p', 'b4p']
R ['b1p', 'b2p', 'b3p', 'b4p'] ['b1q', 'b2q', 'b3q', 'b4q']
U* ['b1q', 'b2q', 'b3q', 'b4q'] ['a1', 'a2', 'b3', 'b4']
U ['c1', 'c2', 'c3', 'c4'] ['c1p', 'c2p', 'c3p', 'c4p']
R ['c1p', 'c2p', 'c3p', 'c4p'] ['c1q', 'c2q', 'c3q', 'c4q']
U* ['c1q', 'c2q', 'c3q', 'c4q'] ['d1', 'c2', 'd3', 'c4']
U ['d1', 'd2', 'd3', 'd4'] ['d1p', 'd2p', 'd3p', 'd4p']
R ['d1p', 'd2p', 'd3p', 'd4p'] ['d1q', 'd2q', 'd3q', 'd4q']
U* ['d1q', 'd2q', 'd3q', 'd4q'] ['c1', 'd2', 'c3', 'd4']
[[['U', 1, 'out', 1], ['R', 1, 'in', 1]], [['U', 1, 'out', 2], ['R', 1, 'in', 2]], [['U', 1, 'out', 3], ['R', 1, 'in', 3]], [['U', 1, 'out', 4], ['R', 1, 'in', 4]], [['R', 1, 'out', 1], ['U*', 1, 'in', 1]], [['R', 1, 'out', 2], ['U*', 1, 'in', 2]], [['R', 1, 'out', 3], ['U*', 1, 'in', 3]], [['R', 1, 'out', 

In [47]:
simplify(sum_)

(d1**3*d2**2*d3**2*d4 + d1**2*d2**3*d3*d4**2 + d1**2*d2*d3**3*d4**2 + 4*d1**2*d2*d3 + d1*d2**2*d3**2*d4**3 + 4*d1*d2**2*d4 + 4*d1*d3**2*d4 + 2*d1*d4 + 4*d2*d3*d4**2 + 2*d2*d3)/(d1**3*d2**3*d3**3*d4**3 + 6*d1**2*d2**2*d3**2*d4**2 + 11*d1*d2*d3*d4 + 6)

In [ ]:
# let d4=1 and d3=d, simplify sum_ ,which should give the average purity of RDM on A.
simplify(sum_.subs({d1:1,d3:1, d4:d/d2}))

(d + d2**2)/(d2*(d + 1))

In [ ]:
# let d3=d2 so |A|=|B|. d1 is intersection dimension, d4 is the remainder dimension.
simplify(sum_.subs({d3:d2}))

-(d1*d2 + d3*d4)/(d1*d2*d3*d4 + 1) - (d1*d3 + d2*d4)/(d1*d2*d3*d4 + 1) + 8*(-d1**2*d2**4*d3**4*d4**2 + d1**2*d2**4*d3**2*d4**2 + d1**2*d2**2*d3**4*d4**2 + 4*d1**2*d2**2*d3**2*d4**2 - 5*d1**2*d2**2*d3**2 - 5*d2**2*d3**2*d4**2 - d2**2*d3**2 + 6*d2**2 + 6*d3**2 - 6)/(d2*d3*(d1**6*d2**6*d3**6*d4**6 - 14*d1**4*d2**4*d3**4*d4**4 + 49*d1**2*d2**2*d3**2*d4**2 - 36)) + (d1**4*d2**6*d3**6*d4**6 - d1**4*d2**6*d3**4*d4**4 - d1**4*d2**4*d3**6*d4**4 + d1**4*d2**4*d3**4*d4**2 - 12*d1**2*d2**4*d3**4*d4**4 + 12*d1**2*d2**4*d3**2*d4**2 + 12*d1**2*d2**2*d3**4*d4**2 + 2*d1**2*d2**2*d3**2*d4**2 - 14*d1**2*d2**2*d3**2 + 22*d2**2*d3**2*d4**2 - 10*d2**2*d3**2 - 12*d2**2 - 12*d3**2 + 12)/(d2*d3*(d1**6*d2**6*d3**6*d4**6 - 14*d1**4*d2**4*d3**4*d4**4 + 49*d1**2*d2**2*d3**2*d4**2 - 36)) + (d1**6*d2**6*d3**6*d4**4 - d1**4*d2**6*d3**4*d4**4 - d1**4*d2**4*d3**6*d4**4 - 12*d1**4*d2**4*d3**4*d4**2 + d1**2*d2**4*d3**4*d4**4 + 12*d1**2*d2**4*d3**2*d4**2 + 12*d1**2*d2**2*d3**4*d4**2 + 2*d1**2*d2**2*d3**2*d4**2 + 22*d1**2*

In [54]:
# covariance: <P(rho_A)P(rho_B)> - <P(rho_A)><P(rho_B)>
from sympy import simplify,factor
factor(simplify(sum_ - (d1 * d2 + d3 * d4)/(d1*d2*d3*d4+1) * (d1*d3 + d2 * d4)/(d1*d2*d3*d4+1)))

2*(d1**2*d2*d3*d4**2 - d1**2*d2*d3 + d1*d2**2*d3**2*d4 - d1*d2**2*d4 - d1*d3**2*d4 + d1*d4 - d2*d3*d4**2 + d2*d3)/((d1*d2*d3*d4 + 1)**2*(d1*d2*d3*d4 + 2)*(d1*d2*d3*d4 + 3))

In [58]:
# variance: A=B, so d2=d3=1,d4 = d/d1, d1 is dim for A, <pur(A)^2> - <pur(A)>^2
factor(simplify(sum_.subs({d2:1, d3:1, d4:d/d1}) - ((d1 + d/d1)/(1 + d))**2 ))

2*(d - d1)*(d + d1)*(d1 - 1)*(d1 + 1)/(d1**2*(d + 1)**2*(d + 2)*(d + 3))